In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df=pd.read_csv('C:/Users/sivanand/Desktop/internship/online_retail_III.csv')
df

,Year,Month,Day,DayOfWeek,IsWeekend,Total_Orders,lag_1,lag_2,lag_7,rolling_7,rolling_3,Total_Revenue
0,2009,12,8,1,0,2189,45083.35,24613.64,54513.50,44590.982857,26500.013333,49517.23
1,2009,12,9,2,0,2433,49517.23,45083.35,63352.51,43877.230000,39738.073333,40616.09
2,2009,12,10,3,0,2403,40616.09,49517.23,74037.91,40629.170000,45072.223333,44442.11
3,2009,12,11,4,0,1439,44442.11,40616.09,40732.92,36401.198571,44858.476667,39659.60
4,2009,12,13,6,1,1767,39659.60,44442.11,9803.05,36247.867143,41572.600000,22313.09
...,...,...,...,...,...,...,...,...,...,...,...,...
592,2011,12,5,0,0,5297,24621.43,57664.07,20611.08,49283.130000,44827.583333,88741.96
593,2011,12,6,1,0,3262,88741.96,24621.43,57165.19,59016.112857,57009.153333,56713.21
594,2011,12,7,2,0,2390,56713.21,88741.96,72595.93,58951.544286,56692.200000,75439.16
595,2011,12,8,3,0,4867,75439.16,56713.21,60126.96,59357.720000,73631.443333,82495.00


In [2]:
# Features — everything except Total_Revenue
X = df.drop('Total_Revenue',axis=1)

# Target — what we want to predict
y = df['Total_Revenue']

print(f"Features shape : {X.shape}")
print(f"Target shape   : {y.shape}")
print(f"\nFeature columns: {X.columns.tolist()}")
print(f"\nSample target values:")
print(y.head())

Features shape : (597, 11)
Target shape   : (597,)

Feature columns: ['Year', 'Month', 'Day', 'DayOfWeek', 'IsWeekend', 'Total_Orders', 'lag_1', 'lag_2', 'lag_7', 'rolling_7', 'rolling_3']

Sample target values:
0    49517.23
1    40616.09
2    44442.11
3    39659.60
4    22313.09
Name: Total_Revenue, dtype: float64


In [3]:
# Last 30 days = test, everything before = train
split_index = len(df) - 30

X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]

print(f"Training days : {len(X_train)}")
print(f"Testing days  : {len(X_test)}")
print(f"\nTraining period: Row 0 → Row {split_index-1}")
print(f"Testing period : Row {split_index} → Row {len(df)-1}")

Training days : 567
Testing days  : 30

Training period: Row 0 → Row 566
Testing period : Row 567 → Row 596


In [4]:
# Scale AFTER splitting 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"✅ Scaling done!")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape : {X_test_scaled.shape}")

✅ Scaling done!
X_train_scaled shape: (567, 11)
X_test_scaled shape : (30, 11)


In [6]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
print("✅ Random Forest trained!")

✅ Random Forest trained!


In [9]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=5,          # limit tree depth!
    min_samples_split=10, # need 10 samples to split
    min_samples_leaf=5,   # need 5 samples in leaf
    random_state=42
)

rf_model.fit(X_train_scaled, y_train)
print("✅ Controlled Random Forest trained!")

y_pred_train = rf_model.predict(X_train_scaled)
y_pred_test  = rf_model.predict(X_test_scaled)

print("\n========== RANDOM FOREST (CONTROLLED) ==========")
print(f"Train R²  : {r2_score(y_train, y_pred_train):.4f}")
print(f"Test  R²  : {r2_score(y_test, y_pred_test):.4f}")
print(f"Train RMSE: £{np.sqrt(mean_squared_error(y_train, y_pred_train)):,.2f}")
print(f"Test  RMSE: £{np.sqrt(mean_squared_error(y_test, y_pred_test)):,.2f}")
print(f"Test  MAE : £{mean_absolute_error(y_test, y_pred_test):,.2f}")

gap = r2_score(y_train, y_pred_train) - r2_score(y_test, y_pred_test)
print(f"\nOverfitting gap: {gap:.4f}")
if gap > 0.1:
    print("⚠️  Overfitting detected!")
else:
    print("✅ No overfitting!")

✅ Controlled Random Forest trained!

========== RANDOM FOREST (CONTROLLED) ==========
Train R²  : 0.7294
Test  R²  : -0.1639
Train RMSE: £9,857.20
Test  RMSE: £21,183.46
Test  MAE : £15,075.15

Overfitting gap: 0.8932
⚠️  Overfitting detected!


In [10]:
# Check your daily_features.csv
df_daily = pd.read_csv('C:/Users/sivanand/Desktop/internship/online_retail_III.csv')

print(f"Shape: {df_daily.shape}")
print(f"\nTotal_Revenue stats:")
print(df_daily['Total_Revenue'].describe())

print(f"\nFirst 10 rows of Total_Revenue:")
print(df_daily[['Year', 'Month', 'Day', 
                'Total_Revenue', 'Total_Orders']].head(10))

print(f"\nTest set (last 30 rows):")
print(df_daily[['Year', 'Month', 'Day', 
                'Total_Revenue']].tail(35))

Shape: (597, 12)

Total_Revenue stats:
count       597.000000
mean      34183.877534
std       19858.192259
min        3457.110000
25%       21948.360000
50%       29283.000000
75%       41145.010000
max      199236.400000
Name: Total_Revenue, dtype: float64

First 10 rows of Total_Revenue:
   Year  Month  Day  Total_Revenue  Total_Orders
0  2009     12    8       49517.23          2189
1  2009     12    9       40616.09          2433
2  2009     12   10       44442.11          2403
3  2009     12   11       39659.60          1439
4  2009     12   13       22313.09          1767
5  2009     12   14       73327.66          4064
6  2009     12   15       50340.24          1973
7  2009     12   16       52792.25          3207
8  2009     12   17       31239.47          1866
9  2009     12   18       42588.09          2033

Test set (last 30 rows):
     Year  Month  Day  Total_Revenue
562  2011     10   31       57369.15
563  2011     11    1       29155.15
564  2011     11    2       4588

In [11]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# TimeSeriesSplit — respects time order
tscv = TimeSeriesSplit(n_splits=5)

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=7,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

# Store scores from each split
r2_scores   = []
rmse_scores = []
mae_scores  = []

print("===== CROSS VALIDATION RESULTS =====")
for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    # Split
    X_tr = X.iloc[train_idx]
    X_te = X.iloc[test_idx]
    y_tr = y.iloc[train_idx]
    y_te = y.iloc[test_idx]

    # Scale
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_te_scaled = scaler.transform(X_te)

    # Train
    rf_model.fit(X_tr_scaled, y_tr)

    # Predict
    y_pred = rf_model.predict(X_te_scaled)

    # Score
    r2   = r2_score(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    mae  = mean_absolute_error(y_te, y_pred)

    r2_scores.append(r2)
    rmse_scores.append(rmse)
    mae_scores.append(mae)

    print(f"Fold {fold}: R²={r2:.4f}  RMSE=£{rmse:,.2f}  MAE=£{mae:,.2f}  "
          f"Train rows={len(train_idx)}  Test rows={len(test_idx)}")

print(f"\n===== AVERAGE ACROSS ALL FOLDS =====")
print(f"Mean R²  : {np.mean(r2_scores):.4f} (+/- {np.std(r2_scores):.4f})")
print(f"Mean RMSE: £{np.mean(rmse_scores):,.2f}")
print(f"Mean MAE : £{np.mean(mae_scores):,.2f}")

===== CROSS VALIDATION RESULTS =====
Fold 1: R²=0.3824  RMSE=£8,181.67  MAE=£6,259.86  Train rows=102  Test rows=99
Fold 2: R²=0.1650  RMSE=£26,297.87  MAE=£14,597.71  Train rows=201  Test rows=99
Fold 3: R²=0.5081  RMSE=£8,639.92  MAE=£6,376.45  Train rows=300  Test rows=99
Fold 4: R²=0.4419  RMSE=£9,979.70  MAE=£6,974.88  Train rows=399  Test rows=99
Fold 5: R²=0.3411  RMSE=£16,545.54  MAE=£11,575.57  Train rows=498  Test rows=99

===== AVERAGE ACROSS ALL FOLDS =====
Mean R²  : 0.3677 (+/- 0.1160)
Mean RMSE: £13,928.94
Mean MAE : £9,156.89
